# 从零实现 Node2Vec：二阶随机游走、Skip-Gram 与时间评估

本 Notebook 用纯 Python/PyTorch 手写 Node2Vec 的二阶 biased walk、skip-gram negative sampling、embedding 训练、链接预测与节点线性评估；不调用 gensim、NetworkX Node2Vec、PyG、DGL 或现成图嵌入实现。

参考：[node2vec, KDD 2016](https://arxiv.org/abs/1607.00653)、[word2vec negative sampling](https://arxiv.org/abs/1310.4546)。合成图结果只用于回归实现与协议，不等价于真实网络上的泛化性能。


In [ ]:
from __future__ import annotations  # 导入本单元所需的依赖。
import copy, hashlib, json, math, random, warnings  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 6201  # 计算并保存当前步骤的中间状态。
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.set_num_threads(1)  # 执行当前语句以推进本节示例。

def digest(payload):  # 定义本节可复用的核心函数。
    return hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(",",":"), ensure_ascii=False).encode()).hexdigest()  # 返回当前分支计算出的结果。

def tensor_desc(state):  # 定义本节可复用的核心函数。
    out={}  # 计算并保存当前步骤的中间状态。
    for k in sorted(state):  # 遍历输入元素以累积或检查结果。
        v=state[k].detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        out[k]={"dtype":str(v.dtype),"shape":list(v.shape),"sha256":hashlib.sha256(v.numpy().tobytes()).hexdigest()}  # 计算并保存当前步骤的中间状态。
    return out  # 返回当前分支计算出的结果。

assert torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
assert digest({"x":1}) != digest({"x":2})  # 用受控断言验证关键不变量。
assert not any(k in globals() for k in ("gensim","networkx","torch_geometric","dgl"))  # 用受控断言验证关键不变量。


## 1. 图合同：有向、无向与时间不能含糊

`TemporalGraph` 的边记录为 `(src,dst,time)`。无向图在邻接表中自动展开两个方向，但语义摘要仍保留规范化后的无向边；有向图只允许沿 `src→dst` 走。重复边、越界节点、非有限时间和自环都 fail closed。

时间切分以 `time <= cutoff` 为训练快照。训练游走绝不能看到未来边；评估负例则从“全时间真边”中过滤，避免把未来会出现的正边当作 false negative。生产中若未来真值未知，应明确报告候选负例可能含潜在正边。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class TemporalGraph:  # 定义承载本节状态与行为的数据结构。
    num_nodes: int  # 执行当前语句以推进本节示例。
    edges: tuple[tuple[int,int,float], ...]  # 执行当前语句以推进本节示例。
    directed: bool = False  # 计算并保存当前步骤的中间状态。

    def validate(self):  # 定义本节可复用的核心函数。
        if not isinstance(self.num_nodes,int) or self.num_nodes < 1 or not self.edges:  # 按当前条件选择后续控制路径。
            raise ValueError("图必须有节点和边")  # 遇到非法合同立即显式失败。
        seen=set()  # 计算并保存当前步骤的中间状态。
        for u,v,t in self.edges:  # 遍历输入元素以累积或检查结果。
            if not (isinstance(u,int) and isinstance(v,int) and 0 <= u < self.num_nodes and 0 <= v < self.num_nodes):  # 按当前条件选择后续控制路径。
                raise ValueError("节点索引非法")  # 遇到非法合同立即显式失败。
            if u == v or not math.isfinite(float(t)):  # 按当前条件选择后续控制路径。
                raise ValueError("拒绝自环或非有限时间")  # 遇到非法合同立即显式失败。
            key=(u,v,t) if self.directed else (min(u,v),max(u,v),t)  # 计算并保存当前步骤的中间状态。
            if key in seen: raise ValueError("重复边")  # 按当前条件选择后续控制路径。
            seen.add(key)  # 执行当前语句以推进本节示例。
        return self  # 返回当前分支计算出的结果。

    def snapshot(self, cutoff: float):  # 定义本节可复用的核心函数。
        self.validate()  # 执行当前语句以推进本节示例。
        kept=tuple(e for e in self.edges if e[2] <= cutoff)  # 计算并保存当前步骤的中间状态。
        if not kept: raise ValueError("快照为空")  # 按当前条件选择后续控制路径。
        return TemporalGraph(self.num_nodes, kept, self.directed)  # 返回当前分支计算出的结果。

    def adjacency(self):  # 定义本节可复用的核心函数。
        self.validate(); adj=[[] for _ in range(self.num_nodes)]  # 计算并保存当前步骤的中间状态。
        for u,v,_ in self.edges:  # 遍历输入元素以累积或检查结果。
            adj[u].append(v)  # 执行当前语句以推进本节示例。
            if not self.directed: adj[v].append(u)  # 按当前条件选择后续控制路径。
        return tuple(tuple(sorted(set(row))) for row in adj)  # 返回当前分支计算出的结果。

    def semantic(self):  # 定义本节可复用的核心函数。
        normalized=sorted((u,v,t) if self.directed else (min(u,v),max(u,v),t) for u,v,t in self.edges)  # 计算并保存当前步骤的中间状态。
        return {"num_nodes":self.num_nodes,"directed":self.directed,"edges":normalized}  # 返回当前分支计算出的结果。

ug=TemporalGraph(4,((0,1,1.),(1,2,2.),(2,3,5.)),False).validate()  # 计算并保存当前步骤的中间状态。
dg=TemporalGraph(4,((0,1,1.),(1,2,2.),(2,3,5.)),True).validate()  # 计算并保存当前步骤的中间状态。
assert ug.adjacency()[1] == (0,2) and dg.adjacency()[1] == (2,)  # 用受控断言验证关键不变量。
assert ug.snapshot(2).adjacency()[2] == (1,)  # 用受控断言验证关键不变量。
assert digest(ug.semantic()) != digest(dg.semantic())  # 用受控断言验证关键不变量。


## 2. Node2Vec 的二阶转移概率

已从 $t$ 走到 $v$，候选下一节点为 $x$。对无向图，本实现使用：

$$\pi_{vx}=w_{vx}\alpha_{pq}(t,x),\quad
\alpha_{pq}(t,x)=\begin{cases}1/p & x=t\\1 & x\in N(t)\\1/q & \text{otherwise}\end{cases}.$$

所有边权为 1。$p<1$ 鼓励立即返回，$q<1$ 鼓励向外探索；`p,q` 必须有限且大于 0。对有向图，“距离 1”明确解释为存在 `t→x`，而不是悄悄对称化。第一次跳转没有前驱，按当前节点邻居均匀采样。


In [ ]:
def transition_probs(adj, previous: int|None, current: int, p: float, q: float):  # 定义本节可复用的核心函数。
    if not (math.isfinite(p) and math.isfinite(q) and p > 0 and q > 0):  # 按当前条件选择后续控制路径。
        raise ValueError("p/q 必须有限且为正")  # 遇到非法合同立即显式失败。
    candidates=adj[current]  # 计算并保存当前步骤的中间状态。
    if not candidates: return (), torch.empty(0)  # 按当前条件选择后续控制路径。
    if previous is None:  # 按当前条件选择后续控制路径。
        weights=torch.ones(len(candidates),dtype=torch.float64)  # 计算并保存当前步骤的中间状态。
    else:  # 处理前置条件不成立的分支。
        previous_neighbors=set(adj[previous]); vals=[]  # 计算并保存当前步骤的中间状态。
        for x in candidates:  # 遍历输入元素以累积或检查结果。
            vals.append(1/p if x == previous else (1.0 if x in previous_neighbors else 1/q))  # 计算并保存当前步骤的中间状态。
        weights=torch.tensor(vals,dtype=torch.float64)  # 计算并保存当前步骤的中间状态。
    return candidates, weights/weights.sum()  # 返回当前分支计算出的结果。

triangle_tail=TemporalGraph(4,((0,1,1.),(1,2,1.),(0,2,1.),(1,3,1.)),False)  # 计算并保存当前步骤的中间状态。
adj=triangle_tail.adjacency()  # 计算并保存当前步骤的中间状态。
cand,prob=transition_probs(adj,0,1,p=2.,q=.5)  # 计算并保存当前步骤的中间状态。
# 当前 1 的候选为 0/2/3：返回 0.5、邻居 1、远点 2，总和 3.5
expected=torch.tensor([.5/3.5,1/3.5,2/3.5],dtype=torch.float64)  # 计算并保存当前步骤的中间状态。
assert cand == (0,2,3) and torch.allclose(prob,expected)  # 用受控断言验证关键不变量。
_, return_heavy=transition_probs(adj,0,1,p=.01,q=10.)  # 计算并保存当前步骤的中间状态。
_, outward_heavy=transition_probs(adj,0,1,p=10.,q=.01)  # 计算并保存当前步骤的中间状态。
assert return_heavy[0] > .98 and outward_heavy[-1] > .98  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    transition_probs(adj,0,1,0,1)  # 执行当前语句以推进本节示例。
    raise AssertionError("p=0 未拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc: assert "p/q" in str(exc)  # 捕获预期异常并验证失败分支。


## 3. 可复现 walk 与孤立点语义

游走每一步都从显式 generator 的 `torch.multinomial` 采样。同 seed、同图、同起点必须逐 token 一致。孤立节点的 walk 只包含起点并立即结束；不能为了凑长度虚构自环，因为那会改变训练分布。

单条 walk 的时间复杂度为 $O(L\bar d)$（此处每步现算候选权重）；生产实现可为每条有向边预计算 alias table，把采样摊还到 $O(1)$，代价是 $O(E)$ 到 $O(E\bar d)$ 的额外索引内存。


In [ ]:
def node2vec_walk(graph: TemporalGraph, start: int, length: int, p: float, q: float,  # 定义本节可复用的核心函数。
                  generator: torch.Generator) -> list[int]:  # 执行当前语句以推进本节示例。
    graph.validate()  # 执行当前语句以推进本节示例。
    if not 0 <= start < graph.num_nodes or length < 1: raise ValueError("start/length 非法")  # 按当前条件选择后续控制路径。
    adj=graph.adjacency(); walk=[start]; previous=None; current=start  # 计算并保存当前步骤的中间状态。
    while len(walk) < length:  # 在终止条件满足前持续推进状态。
        candidates,probs=transition_probs(adj,previous,current,p,q)  # 计算并保存当前步骤的中间状态。
        if not candidates: break  # 按当前条件选择后续控制路径。
        index=int(torch.multinomial(probs.float(),1,generator=generator))  # 计算并保存当前步骤的中间状态。
        nxt=candidates[index]; walk.append(nxt); previous,current=current,nxt  # 计算并保存当前步骤的中间状态。
    return walk  # 返回当前分支计算出的结果。

w1=node2vec_walk(triangle_tail,0,8,1.,1.,torch.Generator().manual_seed(9))  # 计算并保存当前步骤的中间状态。
w2=node2vec_walk(triangle_tail,0,8,1.,1.,torch.Generator().manual_seed(9))  # 计算并保存当前步骤的中间状态。
assert w1 == w2 and len(w1) == 8  # 用受控断言验证关键不变量。
isolated_graph=TemporalGraph(5,triangle_tail.edges,False)  # 计算并保存当前步骤的中间状态。
assert node2vec_walk(isolated_graph,4,10,1,1,torch.Generator().manual_seed(1)) == [4]  # 用受控断言验证关键不变量。
assert node2vec_walk(dg,3,5,1,1,torch.Generator().manual_seed(1)) == [3]  # 用受控断言验证关键不变量。


## 4. `degree^0.75`、无放回且时间一致的负采样

窗口半径为 $w$ 时，每条 walk 产生最多 $O(Lw)$ 个有序 `(center,context)`。本实现令节点负采样权重为训练快照度数的 $d(v)^{3/4}$，对每个正 pair 先过滤 self、正 context 和当前可见真邻居，再按权重无放回抽取 $K$ 个负例；候选或正权重不足时 fail closed，而不是重复同一节点凑数。

时间语义必须分两条路径：训练 sampler 只能读取 `time<=cutoff` 的邻接和度数，未来边可能成为不可避免的潜在假负例；离线评估在标签已经揭晓后，才可用全时间真边过滤候选，避免把 holdout 正边计作负例。把 full graph 提前传给训练 sampler 虽然“更干净”，实质上泄漏了未来标签。


In [ ]:
def walks_to_pairs(walks: list[list[int]], window: int) -> torch.Tensor:  # 定义本节可复用的核心函数。
    if window < 1: raise ValueError("window 必须为正")  # 按当前条件选择后续控制路径。
    pairs=[]  # 计算并保存当前步骤的中间状态。
    for walk in walks:  # 遍历输入元素以累积或检查结果。
        for i,c in enumerate(walk):  # 遍历输入元素以累积或检查结果。
            for j in range(max(0,i-window),min(len(walk),i+window+1)):  # 遍历输入元素以累积或检查结果。
                if i != j: pairs.append((c,walk[j]))  # 按当前条件选择后续控制路径。
    if not pairs: raise ValueError("没有可训练 pair")  # 按当前条件选择后续控制路径。
    return torch.tensor(pairs,dtype=torch.long)  # 返回当前分支计算出的结果。

def degree_weights62(graph: TemporalGraph) -> torch.Tensor:  # 定义本节可复用的核心函数。
    degree = torch.tensor([len(row) for row in graph.adjacency()], dtype=torch.float64)  # 计算并保存当前步骤的中间状态。
    return degree.pow(.75)  # 返回当前分支计算出的结果。

def sample_negatives(centers: torch.Tensor, positive: torch.Tensor, num_nodes: int,  # 定义本节可复用的核心函数。
                     forbidden_neighbors: tuple[frozenset,...], node_weights: torch.Tensor,  # 执行当前语句以推进本节示例。
                     k: int, generator: torch.Generator) -> torch.Tensor:  # 执行当前语句以推进本节示例。
    if centers.shape != positive.shape or centers.dtype != torch.long or k < 1:  # 按当前条件选择后续控制路径。
        raise ValueError("负采样合同非法")  # 遇到非法合同立即显式失败。
    if node_weights.shape != (num_nodes,) or not torch.isfinite(node_weights).all() or bool((node_weights < 0).any()):  # 按当前条件选择后续控制路径。
        raise ValueError("负采样权重非法")  # 遇到非法合同立即显式失败。
    rows=[]  # 计算并保存当前步骤的中间状态。
    for c,pos in zip(centers.tolist(),positive.tolist()):  # 遍历输入元素以累积或检查结果。
        allowed=[n for n in range(num_nodes)  # 计算并保存当前步骤的中间状态。
                 if n != c and n != pos and n not in forbidden_neighbors[c] and float(node_weights[n]) > 0]  # 按当前条件选择后续控制路径。
        if len(allowed) < k:  # 按当前条件选择后续控制路径。
            raise ValueError("过滤后正权重负样本不足，禁止有放回补齐")  # 遇到非法合同立即显式失败。
        weights=node_weights[allowed]  # 计算并保存当前步骤的中间状态。
        chosen=torch.multinomial(weights,k,replacement=False,generator=generator)  # 计算并保存当前步骤的中间状态。
        rows.append(torch.tensor([allowed[i] for i in chosen.tolist()],dtype=torch.long))  # 计算并保存当前步骤的中间状态。
    return torch.stack(rows)  # 返回当前分支计算出的结果。

pairs_probe=walks_to_pairs([[0,1,2]],1)  # 计算并保存当前步骤的中间状态。
assert pairs_probe.tolist() == [[0,1],[1,0],[1,2],[2,1]]  # 用受控断言验证关键不变量。

# 边 (0,3,t=5) 是未来真边，节点 3 在训练快照另有边，因此具备正的 degree^.75 权重。
future_full62=TemporalGraph(5,((0,1,1.),(3,4,1.),(0,3,5.)),False).validate()  # 计算并保存当前步骤的中间状态。
future_train62=future_full62.snapshot(1.)  # 计算并保存当前步骤的中间状态。
train_truth_probe62=tuple(frozenset(row) for row in future_train62.adjacency())  # 计算并保存当前步骤的中间状态。
eval_truth_probe62=tuple(frozenset(row) for row in future_full62.adjacency())  # 计算并保存当前步骤的中间状态。
train_sample_probe62=sample_negatives(torch.tensor([0]),torch.tensor([1]),5,train_truth_probe62,  # 计算并保存当前步骤的中间状态。
                                      degree_weights62(future_train62),2,torch.Generator().manual_seed(3))  # 执行当前语句以推进本节示例。
eval_sample_probe62=sample_negatives(torch.tensor([0]),torch.tensor([1]),5,eval_truth_probe62,  # 计算并保存当前步骤的中间状态。
                                     degree_weights62(future_full62),1,torch.Generator().manual_seed(3))  # 执行当前语句以推进本节示例。
assert 3 in train_sample_probe62[0].tolist()  # 训练不知道未来，不能提前过滤
assert 3 not in eval_sample_probe62[0].tolist()  # 评估已知全时间真值，必须过滤
assert train_sample_probe62.unique().numel()==2  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    sample_negatives(torch.tensor([1]),torch.tensor([0]),3,  # 执行当前语句以推进本节示例。
                     (frozenset({1}),frozenset({0,2}),frozenset({1})),torch.ones(3),1,  # 执行当前语句以推进本节示例。
                     torch.Generator().manual_seed(1))  # 执行当前语句以推进本节示例。
    raise AssertionError("无负例候选未报错")  # 遇到非法合同立即显式失败。
except ValueError as exc: assert "不足" in str(exc)  # 捕获预期异常并验证失败分支。


## 5. 手写 embedding、负采样目标与评估头

Skip-Gram negative sampling 最大化

$$\log\sigma(u_c^\top v_o)+\sum_{k=1}^{K}\log\sigma(-u_c^\top v_{n_k}).$$

输入与上下文 embedding 分开参数化。推理通常取二者平均或只取输入表，本例取输入表。`LinkScorer` 用归一化点积，`NodeLinearProbe` 只在线性评估阶段接收标签。


In [ ]:
class Node2VecSkipGram(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,num_nodes:int,dim:int):  # 定义本节可复用的核心函数。
        super().__init__(); self.input=nn.Embedding(num_nodes,dim); self.context=nn.Embedding(num_nodes,dim)  # 计算并保存当前步骤的中间状态。
        nn.init.normal_(self.input.weight,std=.2); nn.init.zeros_(self.context.weight)  # 计算并保存当前步骤的中间状态。
    def forward(self,center:torch.Tensor,positive:torch.Tensor,negative:torch.Tensor):  # 定义本节可复用的核心函数。
        if center.ndim!=1 or positive.shape!=center.shape or negative.ndim!=2 or len(negative)!=len(center):  # 按当前条件选择后续控制路径。
            raise ValueError("skip-gram batch shape 非法")  # 遇到非法合同立即显式失败。
        u=self.input(center); pos=(u*self.context(positive)).sum(-1)  # 计算并保存当前步骤的中间状态。
        neg=torch.einsum("bd,bkd->bk",u,self.context(negative))  # 计算并保存当前步骤的中间状态。
        return -(F.logsigmoid(pos)+F.logsigmoid(-neg).sum(1)).mean()  # 返回当前分支计算出的结果。
    def embeddings(self): return self.input.weight  # 定义本节可复用的核心函数。

class LinkScorer(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def forward(self,z:torch.Tensor,edges:torch.Tensor):  # 定义本节可复用的核心函数。
        if edges.ndim!=2 or edges.shape[0]!=2: raise ValueError("edges 必须 [2,E]")  # 按当前条件选择后续控制路径。
        zn=F.normalize(z,dim=-1); return (zn[edges[0]]*zn[edges[1]]).sum(-1)  # 计算并保存当前步骤的中间状态。

class NodeLinearProbe(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,dim:int,classes:int): super().__init__(); self.linear=nn.Linear(dim,classes)  # 定义本节可复用的核心函数。
    def forward(self,z:torch.Tensor): return self.linear(z)  # 定义本节可复用的核心函数。

sg_probe=Node2VecSkipGram(4,8)  # 计算并保存当前步骤的中间状态。
assert sg_probe(torch.tensor([0]),torch.tensor([1]),torch.tensor([[2,3]])).ndim==0  # 用受控断言验证关键不变量。
assert LinkScorer()(torch.eye(3),torch.tensor([[0,0],[0,1]])).tolist()==[1.,0.]  # 用受控断言验证关键不变量。
assert NodeLinearProbe(8,2)(torch.zeros(3,8)).shape==(3,2)  # 用受控断言验证关键不变量。


## 6. 受控社区图训练

图由两个社区构成，训练快照内各自接近 clique，并留出两条较晚出现的边做时间链接评估。walk 只在 `cutoff=2` 的快照生成；负例过滤却使用全时间真边。节点分类按节点 ID 划分 train/test，因此只是小型协议测试，真实场景应按未来时间、新节点或整社区做 inductive split。


In [ ]:
edges62=[]  # 计算并保存当前步骤的中间状态。
for base in (0,5):  # 遍历输入元素以累积或检查结果。
    for i in range(base,base+5):  # 遍历输入元素以累积或检查结果。
        for j in range(i+1,base+5):  # 遍历输入元素以累积或检查结果。
            if (i,j) not in ((0,4),(5,9)): edges62.append((i,j,1.0+(i+j)%2))  # 按当前条件选择后续控制路径。
edges62 += [(0,4,3.0),(5,9,3.0)]  # 计算并保存当前步骤的中间状态。
full62=TemporalGraph(10,tuple(edges62),False).validate(); train62=full62.snapshot(2.0)  # 计算并保存当前步骤的中间状态。
walks=[]  # 计算并保存当前步骤的中间状态。
walk_gen=torch.Generator().manual_seed(SEED)  # 计算并保存当前步骤的中间状态。
for _ in range(8):  # 遍历输入元素以累积或检查结果。
    for node in range(10): walks.append(node2vec_walk(train62,node,10,.75,1.5,walk_gen))  # 遍历输入元素以累积或检查结果。
pairs62=walks_to_pairs(walks,2)  # 计算并保存当前步骤的中间状态。
train_truth62=tuple(frozenset(row) for row in train62.adjacency())  # 计算并保存当前步骤的中间状态。
eval_truth62=tuple(frozenset(row) for row in full62.adjacency())  # 计算并保存当前步骤的中间状态。
train_negative_weights62=degree_weights62(train62)  # 计算并保存当前步骤的中间状态。
model62=Node2VecSkipGram(10,12); opt=torch.optim.Adam(model62.parameters(),lr=.03)  # 计算并保存当前步骤的中间状态。
losses=[]  # 计算并保存当前步骤的中间状态。
for step in range(35):  # 遍历输入元素以累积或检查结果。
    idx=torch.randint(len(pairs62),(128,),generator=torch.Generator().manual_seed(SEED+step))  # 计算并保存当前步骤的中间状态。
    batch=pairs62[idx]  # 计算并保存当前步骤的中间状态。
    negatives=sample_negatives(batch[:,0],batch[:,1],10,train_truth62,train_negative_weights62,3,torch.Generator().manual_seed(9000+step))  # 计算并保存当前步骤的中间状态。
    loss=model62(batch[:,0],batch[:,1],negatives)  # 计算并保存当前步骤的中间状态。
    opt.zero_grad(); loss.backward(); opt.step(); losses.append(float(loss))  # 执行当前语句以推进本节示例。

z=model62.embeddings().detach(); probe62=NodeLinearProbe(12,2); probe_opt=torch.optim.Adam(probe62.parameters(),lr=.08)  # 计算并保存当前步骤的中间状态。
labels62=torch.tensor([0]*5+[1]*5); train_nodes=torch.tensor([0,1,2,5,6,7]); test_nodes=torch.tensor([3,4,8,9])  # 计算并保存当前步骤的中间状态。
for _ in range(40):  # 遍历输入元素以累积或检查结果。
    loss=F.cross_entropy(probe62(z[train_nodes]),labels62[train_nodes]); probe_opt.zero_grad(); loss.backward(); probe_opt.step()  # 计算并保存当前步骤的中间状态。
node_acc62=float((probe62(z[test_nodes]).argmax(1)==labels62[test_nodes]).float().mean())  # 计算并保存当前步骤的中间状态。

heldout=torch.tensor([[0,5],[4,9]])  # 计算并保存当前步骤的中间状态。
negative_edges=torch.tensor([[0,1],[9,8]])  # 跨社区非边
link=LinkScorer(); pos_score=link(z,heldout); neg_score=link(z,negative_edges)  # 计算并保存当前步骤的中间状态。
link_pair_acc62=float(((pos_score[:,None] > neg_score[None,:]).float().mean()))  # 计算并保存当前步骤的中间状态。
assert losses[-1] < losses[0] and math.isfinite(losses[-1])  # 用受控断言验证关键不变量。
assert node_acc62 >= .75 and link_pair_acc62 >= .75  # 用受控断言验证关键不变量。
assert not any(t>2 for _,_,t in train62.edges)  # 用受控断言验证关键不变量。
assert 4 not in train_truth62[0] and 4 in eval_truth62[0]  # 用受控断言验证关键不变量。
print({"sg_first":round(losses[0],4),"sg_last":round(losses[-1],4),  # 执行当前语句以推进本节示例。
       "controlled_node_acc":node_acc62,"controlled_link_pair_acc":link_pair_acc62})  # 执行当前语句以推进本节示例。


## 7. 训练/推理复杂度与典型失败

若有 $R$ 条 walk、长度 $L$、窗口 $w$、负例数 $K$、维度 $D$，pair 数约 $O(RLw)$，训练计算约 $O(RLwKD)$。全量 pair 常驻内存会爆炸，生产应流式生成、分片 shuffle、异步 alias sampler，并固定 worker seed。

典型失败：把无向边只存一侧；有向图却偷偷对称化；训练 walk 看见时间 holdout；负采样命中未来真边；孤立点虚构自环；`p/q` 与论文定义相反；同 seed 因多 worker 调度不可复现；链接评估把反向边当负例；transductive 节点结果冒充 inductive 泛化。


## 8. 发布制品与输入语义重算

manifest 绑定 directedness、完整图、训练 cutoff/split、walk 的 `p/q/length/window`、negative filtering recipe 以及 state 的 key/dtype/shape/bytes。`PublishedNode2Vec` 接收真实 `TemporalGraph` 后重新计算其语义，只为注册图返回 embedding。

包内 manifest 即使整体重签，也无法改变只读带外 registry 的 release 摘要。生产实现需把 registry 替换成签名公钥/KMS 与不可变制品仓，并提供吊销和回滚。


In [ ]:
WALK62={"p":.75,"q":1.5,"length":10,"window":2,"walks_per_node":8,"rng":"torch.Generator"}  # 计算并保存当前步骤的中间状态。
SPLIT62={"cutoff":2.0,"train_if":"time<=cutoff","heldout":[[0,4,3.0],[5,9,3.0]]}  # 计算并保存当前步骤的中间状态。
NEG62={"k":3,"weight":"train_snapshot_degree^0.75","replacement":False,"train_filter":"self+positive+train_snapshot_neighbors","evaluation_filter":"self+positive+all-time-neighbors"}  # 计算并保存当前步骤的中间状态。
state62={k:v.detach().cpu().clone() for k,v in model62.state_dict().items()}  # 计算并保存当前步骤的中间状态。
manifest62={"schema":"temporal-edge(src,dst,time)/v1","graph":full62.semantic(),"train_graph":train62.semantic(),  # 计算并保存当前步骤的中间状态。
            "split":SPLIT62,"walk":WALK62,"negative":NEG62,"config":{"num_nodes":10,"dim":12},"state":tensor_desc(state62)}  # 执行当前语句以推进本节示例。
artifact62={"release_id":"node2vec-62-v1","manifest":manifest62,"state":state62}  # 计算并保存当前步骤的中间状态。
def artifact_digest62(a): return digest({"release_id":a["release_id"],"manifest":a["manifest"]})  # 定义本节可复用的核心函数。
_TRUSTED_RELEASES62=MappingProxyType({"node2vec-62-v1":artifact_digest62(artifact62)})  # 计算并保存当前步骤的中间状态。

class PublishedNode2Vec(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,model,graph_semantic): super().__init__(); self.model=model.eval(); self.graph_semantic=graph_semantic  # 定义本节可复用的核心函数。
    def forward(self,graph:TemporalGraph,nodes:torch.Tensor):  # 定义本节可复用的核心函数。
        if digest(graph.validate().semantic()) != digest(self.graph_semantic): raise ValueError("输入图语义未注册")  # 按当前条件选择后续控制路径。
        if nodes.dtype!=torch.long or nodes.ndim!=1 or nodes.numel()==0 or int(nodes.min())<0 or int(nodes.max())>=graph.num_nodes:  # 按当前条件选择后续控制路径。
            raise ValueError("nodes 非法")  # 遇到非法合同立即显式失败。
        with torch.no_grad(): return self.model.embeddings()[nodes]  # 在受管理的上下文中执行操作。

def load62(a):  # 定义本节可复用的核心函数。
    rid=a.get("release_id")  # 计算并保存当前步骤的中间状态。
    if rid not in _TRUSTED_RELEASES62 or artifact_digest62(a)!=_TRUSTED_RELEASES62[rid]: raise ValueError("带外 registry 拒绝 release")  # 按当前条件选择后续控制路径。
    if a["manifest"]["state"] != tensor_desc(a["state"]): raise ValueError("state 描述不匹配")  # 按当前条件选择后续控制路径。
    cfg=a["manifest"]["config"]; m=Node2VecSkipGram(**cfg); m.load_state_dict(a["state"],strict=True)  # 计算并保存当前步骤的中间状态。
    return PublishedNode2Vec(m,a["manifest"]["graph"])  # 返回当前分支计算出的结果。

published62=load62(artifact62)  # 计算并保存当前步骤的中间状态。
assert published62(full62,torch.tensor([0,9])).shape==(2,12)  # 用受控断言验证关键不变量。
attack62=copy.deepcopy(artifact62); attack62["manifest"]["walk"]["p"]=99.; attack62["manifest"]["state"]=tensor_desc(attack62["state"])  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    load62(attack62); raise AssertionError("整体重签攻击未拒绝")  # 执行当前语句以推进本节示例。
except ValueError as exc: assert "registry" in str(exc)  # 捕获预期异常并验证失败分支。
try:  # 尝试执行可能失败的受控操作。
    published62(train62,torch.tensor([0])); raise AssertionError("快照冒充注册图未拒绝")  # 执行当前语句以推进本节示例。
except ValueError as exc: assert "语义" in str(exc)  # 捕获预期异常并验证失败分支。


## 9. 生产检查清单

需要额外覆盖：加权/多重边与方向语义；动态图增量 alias table；高阶节点和孤立节点采样公平性；多进程 RNG；时间一致的负例定义；embedding 漂移与冷启动；ANN 召回误差；多 seed 统计；隐私删除传播；制品签名、词表/节点 ID 映射版本与回滚。

本 Notebook 的结论严格限定为：精确小图转移概率、极端 `p/q`、seed、孤立点、时间泄漏与 false negative oracle 均通过，且受控社区图训练可用；它没有证明参数会迁移到未知业务图。


In [ ]:
assert set(manifest62)=={"schema","graph","train_graph","split","walk","negative","config","state"}  # 用受控断言验证关键不变量。
assert full62.semantic()["directed"] is False  # 用受控断言验证关键不变量。
assert node_acc62 >= .75 and link_pair_acc62 >= .75  # 用受控断言验证关键不变量。
print("Node2Vec 62：转移、时间切分、训练、评估与发布 oracle 通过。")  # 执行当前语句以推进本节示例。
